# Exploratory Data Analysis & Model Prototyping

This notebook covers the reading of historical data for `fact_chromebook_margin_risk`, exploring feature distributions, handling class imbalances, and training the initial MVP Random Forest Classifier.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Simulating the reading of historical data from the DWH
np.random.seed(42)
n_samples = 1000
data = {
    'eur_to_usd': np.random.uniform(1.00, 1.20, n_samples),
    'eur_to_twd': np.random.uniform(32.0, 36.0, n_samples),
    'component_delay_days': np.random.randint(0, 30, n_samples),
    'freight_cost_eur': np.random.uniform(4000, 12000, n_samples),
}
df = pd.DataFrame(data)

# Business Logic: Risk is 1 if cost is high and delay is significant, combined with unfavorable FX
df['margin_impact_risk'] = ((df['freight_cost_eur'] > 10000) & (df['eur_to_twd'] < 33.5) | (df['component_delay_days'] > 20)).astype(int)

df.head()

In [2]:
# Feature distribution checks
df['margin_impact_risk'].value_counts().plot(kind='bar')
plt.title('Target Distribution: Margin Impact Risk')
plt.xlabel('Risk (0=No, 1=Yes)')
plt.ylabel('Count')


In [3]:
# Training MVP model
features = ['eur_to_usd', 'eur_to_twd', 'component_delay_days', 'freight_cost_eur']
target = 'margin_impact_risk'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))